In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import classification_report, f1_score

In [2]:
# !pip install transformers
# !pip install torch

In [2]:
df = pd.read_csv("/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw-pack-2026/texts/hansard500.csv")
display(df.head(2))

,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson


In [6]:
# to use the same label set as part Two

# unify the party name
df["party"] = df["party"].replace("Labour (Co-op)", "Labour")

# filter for the four most popular parties
# The error is because hansard500.The error is because hansard500.csv has so few Liberal Democrat speeches
# after filtering, only 1 remains — not enough for stratified splitting.
# therefore, Liberal Democrat is removed from the dataset
df_cleaned = df[df["party"].isin(["Labour", "Conservative", "Scottish National Party"])]

# remove any rows where the value in the ‘speech class’ column is not ‘Speech’.
df_cleaned = df_cleaned[df_cleaned["speech_class"] == "Speech"]

# remove any rows where the text in the ‘speech’ column is less than 1000 characters long.
df_cleaned = df_cleaned[df_cleaned["speech"].str.len() >= 1000]

print(df_cleaned.shape)
# print(df["party"].value_counts())

(102, 8)


In [7]:
# !pip install huggingface_hub

In [24]:
model_id = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

def zero_shot_classify(speech_text):
    prompt = f"""this is a political speech classifier. 
    Classify the following UK parliamentary speech into exactly one of these parties:
    Conservative, Labour, Scottish National Party.
    
    Output only the party name, nothing else. 
    Speech: {speech_text[:500]}

    Party: """

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=10)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 9263.23it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [31]:
# split the train test dataset without data leakage
speeches = df_cleaned["speech"].values
y = df_cleaned["party"].values

# train, test split on raw text
speech_train, speech_test, y_train, y_test = train_test_split(
    speeches, y, test_size=0.2, random_state=26, stratify=y
)

vectorizer = TfidfVectorizer(stop_words="english", max_features=3000)
X_train = vectorizer.fit_transform(speech_train)
X_test = vectorizer.transform(speech_test)

In [30]:
# run zero-shot on test set
y_pred = [zero_shot_classify(speech) for speech in speech_test]  # raw text, not vectorised
print("macro F1-score:", f1_score(y_test, y_pred, average="macro"))
print("Zero shot Classification Report:")
print(classification_report(y_test, y_pred))

macro F1-score: 0.39999999999999997
Zero shot Classification Report:
                         precision    recall  f1-score   support

           Conservative       0.67      0.67      0.67        12
                 Labour       0.44      0.67      0.53         6
Scottish National Party       0.00      0.00      0.00         3

               accuracy                           0.57        21
              macro avg       0.37      0.44      0.40        21
           weighted avg       0.51      0.57      0.53        21



/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/sisigao/Desktop/Birkbeck_master/Natural_language_processing/0_coursework/cw_venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defin